# Week 5 Lab: Building a Neural Network from Scratch

## Foreword
In this lab, we strip away the mechanics of libraries like PyTorch or TensorFlow.
We will build a simple **2-Layer Neural Network** using only Python and `numpy`.
We will implement:
1.  **Forward Propagation**: Matrix multiplication.
2.  **Activation Functions**: ReLU and Softmax.
3.  **Backpropagation**: Calculating gradients manually.
4.  **Training Loop**: Learning to recognize handwritten digits (MNIST).

### Step 1: Import Dependencies
We need `numpy` for the math and `matplotlib` for viewing digits.
We also import `load_digits` from `sklearn` to easily get the MNIST dataset without dealing with complex binary file parsing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

### Step 2: Prepare the Data
The MNIST dataset contains 8x8 images of handwritten digits (0-9).
1.  Load data.
2.  Normalize pixels (0-16 -> 0.0-1.0).
3.  One-Hot Encode labels (Label '3' becomes `[0,0,0,1,0,0,0,0,0,0]`).

In [ ]:
# Load Data
digits = load_digits()
X = digits.data
y = digits.target

# Normalize
X /= 16.0

# One-Hot Encode Labels
enc = OneHotEncoder(sparse_output=False)
y_onehot = enc.fit_transform(y.reshape(-1, 1))

# Split Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, random_state=42)

print(f"Training Data Shape: {X_train.shape}")
print(f"Test Data Shape: {X_test.shape}")

# Show an example
plt.imshow(X_train[0].reshape(8,8), cmap='gray')
plt.title(f"Label: {np.argmax(y_train[0])}")
plt.show()

### Step 3: Define Activation Functions
We need **Sigmoid** (for simple activation or probabilities) and **Softmax** (for final output probabilities).

In [ ]:
# Exercise 1: Implement Activation Functions

def sigmoid(x):
    # TODO: Implement Sigmoid: 1 / (1 + exp(-x))
    return None 

def softmax(x):
    if x.ndim == 2:
        x = x.T
        x = x - np.max(x, axis=0)
        # TODO: Implement Softmax for batch
        # y = exp(x) / sum(exp(x))
        return None 
    
    x = x - np.max(x)
    # TODO: Implement Softmax for single vector
    return None

<details>
<summary><strong>Click for Solution</strong></summary>

```python
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    if x.ndim == 2:
        x = x.T
        x = x - np.max(x, axis=0)
        y = np.exp(x) / np.sum(np.exp(x), axis=0)
        return y.T 
    x = x - np.max(x)
    return np.exp(x) / np.sum(np.exp(x))```
</details>

### Step 4: Define Loss Function (Cross Entropy)
This measures how different the predicted probabilities are from the true label.
If truth is `[0,1]` and we predicted `[0.1, 0.9]`, error is low. If `[0.9, 0.1]`, error is high.

In [ ]:
# Exercise 2: Implement Loss Function

def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
    batch_size = y.shape[0]
    
    # TODO: Implement Cross Entropy Error
    # Formula: -sum(t * log(y + small_value)) / batch_size
    return None

<details>
<summary><strong>Click for Solution</strong></summary>

```python
def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)
    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size```
</details>

### Step 5: Define Layers (The Building Blocks)
Here we create classes for our layers.
*   **Affine**: Performs $X \cdot W + B$.
*   **Relu**: Performs $max(0, x)$.

In [ ]:
# Exercise 3: Implement Layers

class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        # TODO: Set values <= 0 to 0 using self.mask
        out[self.mask] = 0
        return out

    def backward(self, dout):
        # TODO: Pass gradient only where input was > 0
        dout[self.mask] = 0
        dx = dout
        return dx

class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        # TODO: Implement Affine Forward: dot(x, W) + b
        out = None
        return out

    def backward(self, dout):
        # TODO: Implement Backward (Calculus provided)
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx

<details>
<summary><strong>Click for Solution</strong></summary>

```python
class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0
        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout
        return dx

class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b
        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx```
</details>

### Step 6: Define Output Layer (Softmax + Loss)
This layer combines the final activation and the error calculation for efficiency.

In [ ]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None # Softmax output
        self.t = None # True Label

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size
        return dx

### Step 7: The Network Class (TwoLayerNet)
We assemble the layers into a network.
*   Input Size: 64 (8x8 pixels)
*   Hidden Size: 50 (Arbitrary)
*   Output Size: 10 (Digits 0-9)

In [ ]:
from collections import OrderedDict

class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # Initialize Weights
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        # Generate Layers
        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])

        self.lastLayer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x

    # x: input data, t: teacher data
    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)

    def gradient(self, x, t):
        # Forward
        self.loss(x, t)

        # Backward
        dout = 1
        dout = self.lastLayer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # Save Gradients
        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['b1'] = self.layers['Affine1'].db
        grads['W2'] = self.layers['Affine2'].dW
        grads['b2'] = self.layers['Affine2'].db

        return grads

### Step 8: Training Loop
We train the network for 1000 iterations.

In [ ]:
network = TwoLayerNet(input_size=64, hidden_size=50, output_size=10)

iters_num = 1000
train_size = X_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []

for i in range(iters_num):
    # Get batch
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = X_train[batch_mask]
    t_batch = y_train[batch_mask]

    # Calculate Gradient
    grad = network.gradient(x_batch, t_batch)

    # Update Weights (SGD)
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]

    # Record Loss
    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)
    
    if i % 100 == 0:
        print(f"Iteration {i}: Loss {loss:.4f}")

print("Training Complete")

### Step 9: Visualize Training Progress
We plot the Loss curve. It should go down drastically.

In [ ]:
plt.plot(train_loss_list)
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.show()

### Step 10: Test Accuracy
Let's see how well it works on the test set.

In [ ]:
# Predict Test Set
y_preds = network.predict(X_test)

# Convert One-Hot back to digits
y_pred_digits = np.argmax(y_preds, axis=1)
y_test_digits = np.argmax(y_test, axis=1)

accuracy = np.mean(y_pred_digits == y_test_digits)
print(f"Test Accuracy: {accuracy * 100:.2f}%")